In [23]:
# Library imports
import torch
import math
import random
import json
import nltk
import itertools
import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [3]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 3060 Laptop GPU


In [4]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [5]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
NLTK_MODEL = 'tokenizers/punkt/english.pickle'

In [6]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\erwin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [7]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

c:\Users\erwin\miniconda3\envs\contradetect\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\erwin\.cache\huggingface\hub\models--MoritzLaurer--DeBERTa-v3-large-mnli-fever-anli-ling-wanli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [8]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [35]:
def predict(premise, hypothesis):
    input = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}

def detect(document):
    sentences = nltk.tokenize.sent_tokenize(document, language='english')
    sentence_count = len(sentences)
    combination_count = sentence_count * (sentence_count - 1) // 2

    print(f'Split text into {sentence_count} sentences ({combination_count} combinations)')

    p_contra_max = 0.0
    evidence = []

    for (a, b) in tqdm.tqdm(itertools.combinations(sentences, 2), total=combination_count):
        prediction = predict(a, b)
        p_contra = prediction['contradiction']
        p_contra_max = max(p_contra_max, p_contra)

        if p_contra > 0.75:
            evidence.append((a, b))

    return p_contra_max, evidence

In [20]:
premise = "Why red?"
hypothesis = "Did you see the total lunar eclipse?"

print(predict(premise, hypothesis))

{'entailment': 0.003383298870176077, 'neutral': 0.03746838867664337, 'contradiction': 0.9591482281684875}


In [39]:
pos = False
examples = list(dataset['pos' if pos else 'neg'].values())
example = examples[random.randint(0, len(examples) - 1)]

print(f'Detecting contradictions in example {example['unique id']}')
print()

print(example['text'])
print()

if pos:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence = detect(example['text'])
p_contra, evidence

Detecting contradictions in example wiki_train_28951

 = The Daedalus Variations = 
 
 " The Daedalus Variations " is the 84th episode of the science fiction television series Stargate Atlantis , and is the fourth episode in the series ' fifth season. The episode first aired on August 1 , 2008 on the Sci Fi Channel in the United States , and subsequently aired on October 9 on Sky One in the United Kingdom. The episode was written by Alan McCullough , and directed by regular Stargate director , Andy Mikita. Richard Woolsey ( Robert Picardo ) and Jennifer Keller ( Jewel Staite ) do not appear in the episode , despite being credited during the opening title sequence. The episode bears similarities to the Flying Dutchman , a mythical ghost ship that drifts forever in the ocean with no chance of returning home. It received generally favourable reviews. 
 Episode writer McCullough described the episode as a one - off " wild romp " , with much use of visual effects. The episode follows John S

100%|██████████| 1653/1653 [02:48<00:00,  9.79it/s]


(0.9987623691558838,
 [(' = The Daedalus Variations = \n \n " The Daedalus Variations " is the 84th episode of the science fiction television series Stargate Atlantis , and is the fourth episode in the series \' fifth season.',
   '4 ) , Ghost Hunters International ( with 1.'),
  ('The episode first aired on August 1 , 2008 on the Sci Fi Channel in the United States , and subsequently aired on October 9 on Sky One in the United Kingdom.',
   '4 ) , Ghost Hunters International ( with 1.'),
  ('Richard Woolsey ( Robert Picardo ) and Jennifer Keller ( Jewel Staite ) do not appear in the episode , despite being credited during the opening title sequence.',
   "They use the maneuvering thrusters to stabilize the ship 's orbit above the red giant , and weather another attack by the aliens , with help from an alternate Atlantis."),
  ('Richard Woolsey ( Robert Picardo ) and Jennifer Keller ( Jewel Staite ) do not appear in the episode , despite being credited during the opening title sequence